# Salinity from EN4 data.
Produce single file of EN4 salinity trimmed to Greenland bounding box, for use in bias correction.

16 Mar 2026 | EHU
- 6 May: call EN4 regridding in this notebook and remove from Step2. Automate time labels on output for more flexibility.

### Setup

In [ ]:
SelModel = 'Hadley'

data_directory = f'/Users/eultee/Library/CloudStorage/OneDrive-NASA/Data/Ocean-reanalyses/Hadley/g10'
DirSaveNC   = f'/Users/eultee/Library/CloudStorage/OneDrive-NASA/Data/gris-iceocean-outfiles/'

# Limits of Greenland domain
limN           = 86.0 ## degrees N latitude
limS           = 57.0 ## degrees N latitude
limE           = 4.0 ## degrees E latitude
limW           = 274.0 ## degrees E latitude


In [ ]:
## Regular grid for regridding 
## NOTE: do not edit this without editing the grid in Step2 to match!

# region of interest in EPSG:3413 and format [xmin,xmax,ymin,ymax]
roi = [-7.2e5,9.6e5,-3.45e6,-0.57e6]

# regular grid
hres = 40e3
vres = 50

xreg = np.arange(roi[0],roi[1]+hres,hres)
yreg = np.arange(roi[2],roi[3]+hres,hres)
[Xreg,Yreg] = np.meshgrid(xreg,yreg)
zreg = np.arange(0,2000+vres,vres)

### Imports

In [ ]:
import os
import sys
import glob
import copy
import csv
import numpy as np
import netCDF4 as nc
import xarray as xr
import dask
from datetime import datetime

### Load and trim data
Load in from multiple files. EN4 comes with one NC file per month.  Trim to Greenland bounding box before loading.

In [ ]:
## load all tiles together using multifile dataset -- for EN4
with xr.open_mfdataset(f'{data_directory}/EN*.nc') as ds_temp:
    
    ## trim to Greenland bounding box -- only if not already done
    include_lat = (ds_temp.lat>=limS) & (ds_temp.lat <=limN)
    include_lon = np.logical_or(((ds_temp.lon%360)<=limE),((ds_temp.lon %360) >=limW)) 
    ## modulo 360 to account for lon going -180 to 180 or 0-360
    
    with dask.config.set(**{'array.slicing.split_large_chunks': True}): ## mitigate performance problem with slicing
        gld_ds = ds_temp.where((include_lat & include_lon).compute(), drop=True)
        # ds = gld_ds.load()

gld_ds
    

### Write to a single NetCDF
EN4 has both temperature and salinity in the same dataset. We have used it in the computation of thermal forcing; now we have been requested to provide bias-corrected salinity over the same domain.  Write out salinity to a trimmed single NetCDF to facilitate this.

In [ ]:
## take only salinity from this dataset
sal_out = gld_ds.salinity

In [ ]:
sal_out.assign_attrs(
                    long_name='salinity',
                    # fillvalue=1.1e20,
                    latbounds=[limS, limN],
                    lonbounds=[limW,limE])

In [ ]:
now = datetime.now()
ds_temp = sal_out.to_dataset(name='Salinity')
# ds_temp.TF.attrs = tf_out.attrs
ds_out = ds_temp.assign_attrs(title='Ocean salinity for {}'.format(SelModel),
                             summary='Salinity loaded from reanalysis, trimmed to a' + 
                              ' bounding box around Greenland, for ISMIP7 Greenland forcing.' +
                              ' This version for {}'.format(SelModel),
                             institution='NASA Goddard Space Flight Center',
                             creation_date=now.strftime('%Y-%m-%d %H:%M:%S'))

ds_out

In [ ]:
ds_out.info()

### Write NetCDF out
Write to a custom filename in the directory specified above.  Remember to rename the file as needed.  This automatically tags the date range included in `ds_out`.

In [ ]:
## identify the first and last years for tagging the file
FirstYear = ds_out.time[0].values.item().year
LastYear = ds_out.time[-1].values.item().year

out_fn = DirSaveNC + 'so-EN4-{}_{}.nc'.format(FirstYear, LastYear)

from dask.diagnostics import ProgressBar

with ProgressBar():
    ds_out.to_netcdf(path=out_fn)

### Regrid to prep for QDM
Read the salinity file we just wrote on the EN4 grid.  Brute-force regrid it to a grid matching what we will apply to CMIP in the next step.  This automatically writes to a file with the same name and a `_regrid` tag.

In [ ]:
# file from previous stage
en4Sfile_cropped = out_fn ## assuming we're running this immediately after the above write-out

# do regrid
regrid_en4(en4Sfile_cropped,xreg,yreg,zreg, which_variable='Salinity')

---
### Check the output
This section produces an optional, in-line visualization.  You may run it for a quick check of what you have produced.

In [ ]:
import cartopy  # Map projections libary
import cartopy.crs as ccrs  # Projections list

In [ ]:
f_in = out_fn

ds_new = xr.open_dataset(f_in)
ds_new

In [ ]:
tf_tavg = ds_new.TF.mean(dim='time') 
tf_tavg

In [ ]:
tf_tavg.sel(depth=5.02, method='nearest').mean(skipna=True)

In [ ]:
import matplotlib.pyplot as plt
ax = plt.axes(projection=ccrs.Robinson())
tf_tavg.sel(depth=5.02, method='nearest').plot(ax=ax, transform=ccrs.PlateCarree(), x='lon', y='lat') ## specify x and y coordinates
ax.coastlines(); ax.gridlines();

In [ ]:
tf_tavg.sel(depth=5.02, method='nearest').plot()